# 02 Data Provenance (IEEE 2890-2025)
## Tribal Data Sovereignty Toolkit

## What is provenance?

Provenance is the documented history of a dataset: where it came from,
how it was processed, who is responsible for it, and what obligations
attach to its use.

For Indigenous data, provenance has an additional dimension that standard
scientific data management ignores: **governance provenance** whose land
does the data describe, under what treaty authority was it collected, and
what are the stewardship obligations?

IEEE 2890-2025 establishes a standard for documenting both types of
provenance together, in a machine-readable format that travels with
the data.

In [ ]:
import sys
from pathlib import Path

# Walk up to find repo root
REPO_ROOT = Path().resolve()
while not (REPO_ROOT/"config"/"nation_template.yaml").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        REPO_ROOT = Path().resolve()
        break
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Toolkit root: {REPO_ROOT}")
from toolkit.sovereignty import SovereigntyContext
from toolkit.provenance  import ProvenanceRecord, ProvenanceBuilder, ProvenanceCatalog
ctx = SovereigntyContext.from_config(REPO_ROOT/"config"/"nation_template.yaml")
print("Ready.")

---
## Creating a provenance record

In [ ]:
# Option 1: Direct construction
record = ProvenanceRecord(
    record_id        = "streamflow-pine-ridge-2024",
    dataset_name     = "White River Streamflow for Pine Ridge",
    nation_name      = ctx.config.name,
    territory_name   = ctx.config.territory,
    treaty_reference = ctx.config.treaty_name,
    treaty_status    = ctx.config.treaty_status,
    data_source_name = "USGS National Water Information System",
    data_source_url  = "https://waterdata.usgs.gov/nwis/",
    data_steward     = "US Geological Survey",
    license          = "Public domain",
    classification   = "PUBLIC",
    frameworks       = "OCAP® | CARE | FAIR | IEEE 2890-2025",
    collection_method= "Continuous stream gauge",
    analyst_name     = "Daear Consulting, LLC",
    analysis_purpose = "Drought and water security analysis",
    processing_steps = [
        "Downloaded from USGS NWIS RDB format via API",
        "Parsed daily mean discharge (parameter 00060, statistic 00003)",
        "Computed 7-day minimum flow per water year",
        "Classified by drought stage (Normal / Watch / Emergency)",
    ],
    notes = (
        "USGS streamflow gauge coverage on Pine Ridge Reservation is "
        "sparse. Four gauges on the White River system provide incomplete "
        "coverage. Monitoring gaps are a federal infrastructure equity finding."
    ),
)

print("Provenance record created:")
print(f"  ID      : {record.record_id}")
print(f"  Nation  : {record.nation_name}")
print(f"  Source  : {record.data_source_name}")
print(f"  Class   : {record.classification}")

In [ ]:
# Option 2: Fluent builder (recommended for integration with SovereigntyContext)
record2 = (
    ProvenanceBuilder("ssurgo-pine-ridge-2024")
    .for_nation(ctx)
    .for_dataset("SSURGO Soil Survey for Pine Ridge and Rosebud")
    .from_source("usda_ssurgo")
    .collected_by("USDA NRCS", method="Field survey and laboratory analysis")
    .processed_by("Daear Consulting, LLC",
                  purpose="Soil health and land capability analysis")
    .step("Downloaded via ESRI Soil Data Downloader by county AREASYMBOL")
    .step("Loaded map unit polygons (MUPOLYGON layer) from file geodatabase")
    .step("Joined component and horizon tables on mukey and cokey")
    .step("Computed linear extensibility percent (LEP) per map unit")
    .classify("PUBLIC")
    .note("SSURGO sampling density on Tribal lands is lower than adjacent "
          "agricultural counties. This is a federal investment equity gap, "
          "not evidence of uniform soil conditions.")
    .build()
)

print("Record built via fluent builder:")
print(record2.to_markdown()[:600])
print("...")

In [ ]:
# Export provenance records
from pathlib import Path

outputs = Path(REPO_ROOT) / "outputs"
outputs.mkdir(exist_ok=True)

# Save as JSON (machine-readable, IEEE 2890-2025 compatible)
record.save(outputs/"provenance_streamflow.json", format="json")

# Save as Markdown (human-readable, for data packages and README files)
record2.save(outputs/"provenance_ssurgo.md", format="markdown")

print("Saved:")
print(f"  {outputs/'provenance_streamflow.json'}")
print(f"  {outputs/'provenance_ssurgo.md'}")

---
## Building a project-level provenance catalog

In [ ]:
# A provenance catalog collects all records for a project
catalog = ProvenanceCatalog("Tribal Water Monitoring for Pine Ridge 2024")
catalog.add(record)
catalog.add(record2)

# Add more records as needed
tribal_record = (
    ProvenanceBuilder("tribal-well-logs-2024")
    .for_nation(ctx)
    .for_dataset("Tribal Well Log Data")
    .from_source("tribal_field_data")
    .processed_by("Natural Resources Department",
                  purpose="Aquifer characterization")
    .step("Collected by Tribal staff using well_log_template.xlsx")
    .step("Verified by licensed hydrologist")
    .classify("GITIGNORED")
    .note("OCAP® governed: never commit to version control")
    .build()
)
catalog.add(tribal_record)

catalog.print_summary()
catalog.save(outputs/"provenance_catalog.json")

## Attaching provenance to a GeoDataFrame

In [ ]:
# Provenance attributes can be attached directly to a GeoDataFrame
# so they travel with the data through all processing steps

import geopandas as gpd
from shapely.geometry import Point
import pandas as pd

# Simulated streamflow gauge locations
gauges = gpd.GeoDataFrame({
    "site_no":   ["06446000", "06446500", "06447000"],
    "name":      ["White River Near Oglala SD",
                  "White River Near Interior SD",
                  "White River Near Kadoka SD"],
    "flow_cfs":  [12.4, 8.1, 15.2],
    "geometry":  [Point(-102.75, 43.18),
                  Point(-101.98, 43.73),
                  Point(-101.52, 43.83)],
}, crs="EPSG:4326")

# Attach provenance so that every row carries sovereignty metadata
gauges_with_prov = ctx.attach_provenance(
    gauges,
    source_key     = "usgs_nwis_streamflow",
    classification = "PUBLIC",
    additional     = {"analysis_date": "2024-06-01"}
)

print("Columns after provenance attachment:")
prov_cols = [c for c in gauges_with_prov.columns if c.startswith("prov_")]
for col in prov_cols:
    val = gauges_with_prov[col].iloc[0]
    print(f"  {col:<30}: {str(val)[:50]}")